# LOGOS — Fast GPU Training (5MB TinyStories)
**Vedic GEMM + Langevin Dynamics | NO Adam**

| Config | Value |
|--------|-------|
| Optimizer | Langevin Dynamics |
| Dataset | 5MB TinyStories |
| d_model | 64 |
| Layers | 2 |
| Heads | 4 |
| Epochs | 10 |
| Expected time | ~3-5 min on T4 |

In [ ]:
# CELL 1 — Imports + GPU Check
import os, subprocess, re, math, shutil, time, glob, json
import matplotlib.pyplot as plt

WORK_DIR   = '/kaggle/working/LOGOS'
TRAIN_FILE = '/kaggle/working/dataset.txt'
LOG_FILE   = '/kaggle/working/training_log.txt'

print('=== GPU ===')
os.system('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')
print('\n=== CUDA ===')
os.system('nvcc --version | grep release')
print('\n=== GCC ===')
os.system('g++ --version | head -1')
print('\n=== Disk ===')
os.system('df -h /kaggle/working | tail -1')
print('\ndone')

In [ ]:
# CELL 2 — Dataset: 5MB TinyStories
TARGET_MB    = 5
TARGET_BYTES = TARGET_MB * 1024 * 1024

if os.path.exists(TRAIN_FILE) and os.path.getsize(TRAIN_FILE) >= TARGET_BYTES // 2:
    sz = os.path.getsize(TRAIN_FILE) // 1024
    print(f'dataset.txt already exists: {sz} KB — reusing')
    with open(TRAIN_FILE) as f:
        print(f.read(200))
else:
    done = False

    # 1. Any large local .txt
    for fp in glob.glob('/kaggle/input/**/*.txt', recursive=True):
        if os.path.getsize(fp) > 512*1024:
            print(f'Using local: {fp}')
            with open(fp, 'r', errors='ignore') as fin, open(TRAIN_FILE, 'w') as fout:
                fout.write(fin.read(TARGET_BYTES))
            done = True; break

    # 2. Any local .jsonl
    if not done:
        for fp in glob.glob('/kaggle/input/**/*.jsonl', recursive=True):
            if os.path.getsize(fp) > 512*1024:
                print(f'Using JSONL: {fp}')
                written = 0
                with open(TRAIN_FILE, 'w') as fout:
                    with open(fp, 'r', errors='ignore') as fin:
                        for line in fin:
                            if written >= TARGET_BYTES: break
                            try:
                                obj = json.loads(line)
                                t = obj.get('story', obj.get('text', obj.get('content','')))
                                if t:
                                    fout.write(t.strip() + '\n\n')
                                    written += len(t)
                            except: pass
                done = True; break

    # 3. Download 5MB from HuggingFace
    if not done:
        print('Downloading from HuggingFace (5MB)...')
        tmp = '/kaggle/working/_tmp.txt'
        ret = subprocess.run(
            ['wget', '-q', '--timeout=60',
             'https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt',
             '-O', tmp],
            capture_output=True)
        if ret.returncode == 0 and os.path.exists(tmp):
            with open(tmp,'r',errors='ignore') as fin, open(TRAIN_FILE,'w') as fout:
                fout.write(fin.read(TARGET_BYTES))
            os.remove(tmp)
            done = True
            print('Downloaded ok')

    # 4. Synthetic fallback
    if not done:
        print('Using synthetic fallback...')
        stories = [
            'Once upon a time there was a little girl named Lily. She loved to play in the garden with her dog Max.\n\n',
            'Tom was a curious boy who loved reading books about science and nature every evening.\n\n',
            'The sun rose slowly over the small village. Birds began to sing their morning songs.\n\n',
            'Ben and his sister Sara found a tiny puppy near the old oak tree in the park.\n\n',
            'Emma wanted to bake a cake for her mothers birthday. She mixed flour eggs and sugar carefully.\n\n',
            'Jake loved to draw pictures of dragons and castles in his notebook during class.\n\n',
            'Maya and her grandfather went fishing by the river every Saturday morning.\n\n',
            'The old wizard lived in a tall tower. He spent his days studying ancient maps.\n\n',
        ]
        with open(TRAIN_FILE, 'w') as f:
            written = 0
            while written < TARGET_BYTES:
                for s in stories:
                    f.write(s); written += len(s)

sz = os.path.getsize(TRAIN_FILE)
print(f'Dataset: {sz//1024} KB')
print('Preview:', open(TRAIN_FILE).read(200))

In [ ]:
# CELL 3 — Clone / Pull LOGOS
REPO_URL = 'https://github.com/Vikas8719/LOGOS.git'

if not os.path.exists(WORK_DIR):
    print('Cloning...')
    os.system(f'git clone {REPO_URL} {WORK_DIR}')
else:
    print('Pulling...')
    os.system(f'git -C {WORK_DIR} pull origin main')

os.chdir(WORK_DIR)
os.system('git log --oneline -3')
os.system('ls src/ cuda/')

In [ ]:
# CELL 4 — Patch main.cpp: d=64 L=2 H=4 seq=64 epochs=10
MAIN_CPP = f'{WORK_DIR}/src/main.cpp'
with open(MAIN_CPP) as f:
    src = f.read()

src = re.sub(r'cfg\.d_model\s*=\s*\d+;',    'cfg.d_model     = 64;',  src)
src = re.sub(r'cfg\.num_layers\s*=\s*\d+;',  'cfg.num_layers  = 2;',   src)
src = re.sub(r'cfg\.num_heads\s*=\s*\d+;',   'cfg.num_heads   = 4;',   src)
src = re.sub(r'cfg\.max_seq_len\s*=\s*\d+;', 'cfg.max_seq_len = 64;',  src)
src = re.sub(r'int SEQ\s*=\s*\d+;',          'int SEQ   = 64;',        src)
src = re.sub(r'int EPOCHS\s*=\s*\d+;',       'int EPOCHS = 10;',       src)
src = re.sub(r'LangevinOptimizer langevin\([\de.+-]+f',
             'LangevinOptimizer langevin(5e-4f', src)

with open(MAIN_CPP, 'w') as f:
    f.write(src)

print('Patched. Checking key values:')
for line in src.split('\n'):
    for kw in ['d_model','num_layers','num_heads','int SEQ','int EPOCHS','LangevinOptimizer langevin']:
        if kw in line and '=' in line and not line.strip().startswith('//'):
            print(' ', line.strip())
            break

In [ ]:
# CELL 5 — Build
os.chdir(WORK_DIR)
os.system('rm -rf build && mkdir build')

try:
    r = subprocess.run(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader'],
                       capture_output=True, text=True)
    cap = r.stdout.strip().replace('.', '')
except:
    cap = '75'
print(f'GPU arch: sm_{cap}')

os.system(f'''
cd {WORK_DIR} &&
cmake -B build -DCMAKE_BUILD_TYPE=Release \
  -DCMAKE_CXX_FLAGS="-O3 -march=native -std=c++20" \
  -DCMAKE_CUDA_ARCHITECTURES="{cap}" 2>&1 | tail -5 &&
cmake --build build --parallel $(nproc) 2>&1
''')

os.system(f'ls -lh {WORK_DIR}/build/logos* 2>/dev/null || echo "No binary"')

GPU_BIN = f'{WORK_DIR}/build/logos_gpu'
CPU_BIN = f'{WORK_DIR}/build/logos'
BINARY  = GPU_BIN if os.path.exists(GPU_BIN) else CPU_BIN
print(f'Using: {BINARY}')

In [ ]:
# CELL 6 — TRAIN (5MB, ~3-5 min)
os.chdir(WORK_DIR)
GPU_BIN = f'{WORK_DIR}/build/logos_gpu'
CPU_BIN = f'{WORK_DIR}/build/logos'
BINARY  = GPU_BIN if os.path.exists(GPU_BIN) else CPU_BIN
MODE    = 'GPU' if os.path.exists(GPU_BIN) else 'CPU'

print(f'Mode: {MODE} | Data: {os.path.getsize(TRAIN_FILE)//1024}KB | d=64 L=2 H=4 epochs=10')
print('Langevin: dW = -gamma*grad*lr + sqrt(2*gamma*T)*noise')
print()

steps_log, losses_log, smooth_log = [], [], []
smooth = -1
start_t = time.time()

proc = subprocess.Popen(
    [BINARY, '--train', TRAIN_FILE],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    universal_newlines=True, bufsize=1, cwd=WORK_DIR
)

with open(LOG_FILE, 'w') as log:
    try:
        for line in proc.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
            m = re.search(r'Step\s+(\d+).*Loss:\s+([\d.]+)', line)
            if m:
                s, l = int(m.group(1)), float(m.group(2))
                if not (math.isnan(l) or l > 30):
                    steps_log.append(s)
                    losses_log.append(l)
                    smooth = l if smooth < 0 else 0.95*smooth + 0.05*l
                    smooth_log.append(smooth)
    except KeyboardInterrupt:
        proc.terminate()
        print('Stopped')

proc.wait()
elapsed = time.time() - start_t
print(f'Time: {elapsed/60:.1f} min')
if losses_log:
    drop = losses_log[0] - losses_log[-1]
    print(f'Start: {losses_log[0]:.4f} | Final: {losses_log[-1]:.4f} | Drop: {drop:.4f}')
    if drop > 1.0:   print('Model IS learning!')
    elif drop > 0.2: print('Slight drop — increase epochs')
    else:            print('No drop — check patch + build output')

In [ ]:
# CELL 7 — Loss Plot
if not steps_log and os.path.exists(LOG_FILE):
    smooth = -1
    with open(LOG_FILE) as f:
        for line in f:
            m = re.search(r'Step\s+(\d+).*Loss:\s+([\d.]+)', line)
            if m:
                s, l = int(m.group(1)), float(m.group(2))
                if not (math.isnan(l) or l > 30):
                    steps_log.append(s); losses_log.append(l)
                    smooth = l if smooth<0 else 0.95*smooth+0.05*l
                    smooth_log.append(smooth)

if len(steps_log) < 2:
    print('Run Cell 6 first')
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('LOGOS — Langevin Dynamics Training', fontsize=13, fontweight='bold')

    ax1.plot(steps_log, losses_log, 'steelblue', lw=0.7, alpha=0.4, label='Raw')
    ax1.plot(steps_log, smooth_log, 'crimson', lw=2, label='Smoothed')
    ax1.axhline(losses_log[0],  color='orange', ls='--', lw=1, label=f'Start={losses_log[0]:.2f}')
    ax1.axhline(losses_log[-1], color='green',  ls='--', lw=1, label=f'Final={losses_log[-1]:.2f}')
    ax1.set_title('Cross-Entropy Loss'); ax1.set_xlabel('Steps')
    ax1.legend(); ax1.grid(alpha=0.3)

    perp = [math.exp(min(l, 12)) for l in smooth_log]
    ax2.plot(steps_log, perp, 'darkorchid', lw=1.5)
    ax2.fill_between(steps_log, perp, alpha=0.1, color='darkorchid')
    ax2.set_yscale('log')
    ax2.set_title(f'Perplexity: {perp[0]:.0f} to {perp[-1]:.0f}')
    ax2.set_xlabel('Steps'); ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('/kaggle/working/loss_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Loss drop: {losses_log[0]:.4f} to {losses_log[-1]:.4f}')

In [ ]:
# CELL 8 — Generate Text
os.chdir(WORK_DIR)
CPU_BIN = f'{WORK_DIR}/build/logos'
ckpts   = sorted([f for f in glob.glob(f'{WORK_DIR}/*.bin') if 'vocab' not in f])
print('Checkpoints:', [os.path.basename(c) for c in ckpts])
if ckpts:
    latest = ckpts[-1]
    for p in ['Once upon a time', 'The little girl', 'Tom and his dog']:
        print(f"\nPrompt: '{p}'")
        print('-'*40)
        os.system(f'{CPU_BIN} --generate {latest} "{p}" 2>&1')
else:
    print('No checkpoint — run Cell 6 first')